# Step 1: Nighttime Lights Before and After Typhoon Haiyan (NASA Black Marble)

**Goals**
1. Figure 1: maps of nighttime lights before and after Haiyan's landfall in the central Philippines, plus a "percent of normal" map
2. Figure 2: weekly nighttime-light recovery trajectories for Tacloban, Ormoc and Cebu City (control), with a simple recovery summary table

**Processing choices**
1. **Quality and cloud screening**: keep only high-quality retrievals (`Mandatory_Quality_Flag` 0 or 1), then drop pixels flagged "probably cloudy" or "confidently cloudy" in `QF_Cloud_Mask`
2. **Coverage filter**: drop a whole night when fewer than half of the pixels in a city's buffer are valid (partly clouded nights are the main source of spurious dips)
3. **Long pre-event baseline**: median of clear nights from 1 July to 7 November 2013; the number of nights used is reported
4. **Weekly aggregation**: 7-day bins anchored on the day after landfall; a weekly median is computed only when at least 2 clear nights are available
5. **Previous-year reference**: the same city over the same weeks one year earlier (no typhoon), to separate the typhoon's effect from seasonal change (monsoon clouds, holiday lighting)
6. **Clear-night counts**: bars under each trajectory show how many usable nights each week has, so data gaps are visible at a glance
7. **Robust recovery metrics**: light levels at weeks 2, 4, 8, 12 and 16, and the first week at which the weekly median stays at or above the threshold for two consecutive weeks

**Before running**
- Register at <https://code.earthengine.google.com/register> (choose *Unpaid usage → Academia & Research*), create a Cloud project and note its ID (e.g. `ee-yourname`)
- In Colab: *File → Upload notebook*, then run the cells from top to bottom (Shift + Enter)
- Replace `PROJECT_ID` in cell 2 with your own project ID

**Timing note**: Haiyan made landfall at Guiuan, Eastern Samar, early on 8 November 2013 (local time). The VIIRS overpass is around 01:30 local time, so the 8 November image is still pre-landfall; post-event windows start on 9 November.

In [ ]:
# 1. Install dependencies (Colab usually ships earthengine-api already)
!pip install -q earthengine-api geemap

In [ ]:
# 2. Authenticate and initialize Earth Engine
import ee

PROJECT_ID = "ee-yourname"   # replace with your own Cloud project ID

ee.Authenticate()            # follow the link to authorize
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialized")

## Parameters

For a different typhoon, this is usually the only cell you need to change.

In [ ]:
# 3. Study parameters
import pandas as pd

LANDFALL = "2013-11-08"
PRE_WINDOW  = ("2013-07-01", "2013-11-07")     # pre-event baseline: ~4 months
POST_WINDOW = ("2013-11-09", "2013-11-23")     # post-event window for the maps: 2 weeks
TS_WINDOW   = ("2013-07-01", "2014-04-01")     # time-series range

# Previous-year reference (no typhoon): shift everything back 52 weeks
SHIFT = pd.Timedelta(weeks=52)
shift = lambda w: tuple((pd.Timestamp(d) - SHIFT).strftime("%Y-%m-%d") for d in w)
PRE_WINDOW_PRIOR, TS_WINDOW_PRIOR = shift(PRE_WINDOW), shift(TS_WINDOW)

# Map extent: Samar, Leyte, northern Cebu to eastern Panay  [west, south, east, north]
REGION_BOUNDS = [121.8, 9.8, 126.0, 12.8]
REGION = ee.Geometry.Rectangle(REGION_BOUNDS)

# Two heavily hit cities + one less-affected control city
PLACES = {
    "Tacloban": (125.0036, 11.2447),
    "Ormoc": (124.6075, 11.0064),
    "Cebu City (control)": (123.8854, 10.3157),
}
BUFFER_M = 3000           # average within a 3 km radius of each city centre

USE_CLOUD_MASK = True     # drop pixels flagged probably/confidently cloudy
MIN_VALID_FRAC = 0.5      # a night needs >= 50% of the buffer's pixels to be valid
MIN_DAYS_PER_WEEK = 2     # a week needs >= 2 clear nights to get a median
RECOVERY_LEVEL = 80       # recovered = >= 80% of baseline for 2 consecutive weeks
print("Previous-year reference window:", TS_WINDOW_PRIOR)

In [ ]:
# 4. Load data: quality flag + cloud mask
COLLECTION = "NASA/VIIRS/002/VNP46A2"
NTL_BAND = "DNB_BRDF_Corrected_NTL"
QF_BAND = "Mandatory_Quality_Flag"
CLOUD_BAND = "QF_Cloud_Mask"

col = ee.ImageCollection(COLLECTION)
bands = col.first().bandNames().getInfo()
print("Available bands:", bands)
assert NTL_BAND in bands and QF_BAND in bands, "Unexpected band names - check the band list printed above"
if USE_CLOUD_MASK and CLOUD_BAND not in bands:
    print("Warning: cloud-mask band not found; screening with the quality flag only")
    USE_CLOUD_MASK = False

def clean(img):
    # Mandatory_Quality_Flag: 0/1 = high quality, 2 = poor (possible cloud contamination), 255 = fill
    ok = img.select(QF_BAND).lte(1)
    if USE_CLOUD_MASK:
        # QF_Cloud_Mask bits 6-7: 0 confident clear, 1 probably clear, 2 probably cloudy, 3 confident cloudy
        cloud = img.select(CLOUD_BAND).rightShift(6).bitwiseAnd(3)
        ok = ok.And(cloud.lte(1))
    return (img.select(NTL_BAND).updateMask(ok).rename("ntl")
               .copyProperties(img, ["system:time_start"]))

ntl = col.map(clean)

## Maps: before, after, and percent of normal

- Before / after: median of clear nights in each window
- Percent of normal = after ÷ before × 100, computed only where there was meaningful light before the event
- "Meaningful light" uses a relative threshold (10% of the region's 99th-percentile pre-event radiance), so it does not depend on units

In [ ]:
# 5. Composites
pre = ntl.filterDate(*PRE_WINDOW).median().rename("pre")
post = ntl.filterDate(*POST_WINDOW).median().rename("post")

p99 = pre.reduceRegion(ee.Reducer.percentile([99]), REGION, 500,
                       maxPixels=1e9, bestEffort=True).get("pre").getInfo()
LIT_THRESHOLD = 0.1 * p99
lit = pre.gte(LIT_THRESHOLD)
pct = post.divide(pre).multiply(100).updateMask(lit).rename("pct")
print(f"p99 = {p99:.2f}; lit threshold = {LIT_THRESHOLD:.2f}")

In [ ]:
# 6. Static map figure
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm, TwoSlopeNorm

INK, INK2, SURFACE, GRID = "#1f1f1e", "#5f5e5a", "#fcfcfb", "#e5e4e0"
plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.edgecolor": "#c9c8c4",
                     "figure.facecolor": SURFACE, "axes.facecolor": SURFACE})
os.makedirs("figures", exist_ok=True)

RES = 0.005
xmin, ymin, xmax, ymax = REGION_BOUNDS
W, H = int(round((xmax - xmin) / RES)), int(round((ymax - ymin) / RES))
arr = ee.data.computePixels({
    "expression": pre.addBands(post).addBands(pct).unmask(-1),
    "fileFormat": "NUMPY_NDARRAY",
    "grid": {"dimensions": {"width": W, "height": H},
             "affineTransform": {"scaleX": RES, "shearX": 0, "translateX": xmin,
                                 "shearY": 0, "scaleY": -RES, "translateY": ymax},
             "crsCode": "EPSG:4326"},
})
def band(name):
    a = arr[name].astype(float); a[a < 0] = np.nan; return a
A_pre, A_post, A_pct = band("pre"), band("post"), band("pct")
litmask = A_pre >= LIT_THRESHOLD
A_pre[~litmask] = np.nan; A_post[~litmask] = np.nan

seq = LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#6da7ec", "#256abf", "#0d366b"])
div = LinearSegmentedColormap.from_list("div", ["#b8302f", "#e34948", "#f0a3a2", "#f0efec",
                                                "#86b6ef", "#2a78d6", "#104281"])
for cm in (seq, div):
    cm.set_bad(SURFACE)

lognorm = LogNorm(vmin=max(np.nanpercentile(A_pre, 5), 1e-3), vmax=p99)
fig, axes = plt.subplots(1, 3, figsize=(15, 5.6), constrained_layout=True)
panels = [
    (A_pre, seq, lognorm, f"Before: {PRE_WINDOW[0]} to {PRE_WINDOW[1]}", "Radiance (log scale)"),
    (A_post, seq, lognorm, f"After: {POST_WINDOW[0]} to {POST_WINDOW[1]}", "Radiance (log scale)"),
    (A_pct, div, TwoSlopeNorm(vcenter=100, vmin=0, vmax=200), "After ÷ before", "Percent of normal (%)"),
]
for ax, (data, cmap, norm, title, clabel) in zip(axes, panels):
    im = ax.imshow(data, extent=[xmin, xmax, ymin, ymax], cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_title(title, loc="left", fontsize=11); ax.set_xlabel("Longitude"); ax.set_aspect("equal")
    cb = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
    cb.set_label(clabel, color=INK2); cb.outline.set_visible(False)
    if isinstance(norm, LogNorm):
        nice = [v for v in (0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000) if norm.vmin <= v <= norm.vmax]
        cb.set_ticks(nice); cb.set_ticklabels([f"{v:g}" for v in nice]); cb.ax.minorticks_off()
    for label, (lon, lat) in PLACES.items():
        ax.plot(lon, lat, "o", ms=8, mfc="none", mec=INK, mew=1.5)
        ax.annotate(label.split(" (")[0], (lon, lat), xytext=(6, 6), textcoords="offset points", fontsize=9)
axes[0].set_ylabel("Latitude")
fig.suptitle("Nighttime lights before and after Typhoon Haiyan (2013), central Philippines · NASA Black Marble VNP46A2",
             x=0.01, ha="left", fontsize=12)
fig.text(0.01, -0.02, "Blank = no meaningful pre-event light or no clear observation. "
         "Gray ≈ unchanged; red = darker than before.", fontsize=9, color=INK2)
fig.savefig("figures/fig1_haiyan_ntl_maps.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Lit pixels below 50% of normal after landfall: {np.nanmean(A_pct[litmask] < 50) * 100:.0f}%")

## Time series: daily observations → screening → weekly medians

For each city we extract two periods: the typhoon year and the same weeks one year earlier (reference).

In [ ]:
# 7. Extract nightly mean radiance and valid-pixel count per city
def place_series(name, lon, lat, window, period):
    geom = ee.Geometry.Point([lon, lat]).buffer(BUFFER_M)
    def per_image(img):
        s = img.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True), geom, 500)
        return ee.Feature(None, {"date": img.date().format("YYYY-MM-dd"),
                                 "ntl": s.get("ntl_mean"), "n": s.get("ntl_count")})
    rows = [f["properties"] for f in ntl.filterDate(*window).map(per_image).getInfo()["features"]]
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    df["place"], df["period"] = name, period
    return df

frames = []
for name, (lon, lat) in PLACES.items():
    frames.append(place_series(name, lon, lat, TS_WINDOW, "event"))
    frames.append(place_series(name, lon, lat, TS_WINDOW_PRIOR, "prior"))
raw = pd.concat(frames, ignore_index=True)
raw["ntl"] = pd.to_numeric(raw["ntl"], errors="coerce")
raw["n"] = pd.to_numeric(raw["n"], errors="coerce").fillna(0)

In [ ]:
# 8. Coverage filter + pre-event baseline + weekly aggregation
landfall = pd.Timestamp(LANDFALL)

def weekly_series(df, pre_window, anchor):
    df = df.sort_values("date").copy()
    full_n = df["n"].max()
    df["valid"] = df["ntl"].notna() & (df["n"] >= MIN_VALID_FRAC * full_n)
    base_days = df[df["valid"] & df["date"].between(*pre_window)]
    base = base_days["ntl"].median()
    df["pct"] = np.where(df["valid"], df["ntl"] / base * 100, np.nan)
    # week 0 starts the day after landfall
    df["week"] = np.floor((df["date"] - (anchor + pd.Timedelta(days=1))).dt.days / 7).astype(int)
    wk = df.groupby("week").agg(pct=("pct", "median"), days=("valid", "sum")).reset_index()
    wk.loc[wk["days"] < MIN_DAYS_PER_WEEK, "pct"] = np.nan
    wk["mid"] = anchor + pd.Timedelta(days=1) + pd.to_timedelta(wk["week"] * 7 + 3, unit="D")
    return df, wk, base, len(base_days)

daily, weekly, summary = {}, {}, []
for name in PLACES:
    d_e, w_e, base, n_base = weekly_series(raw.query("place == @name and period == 'event'"), PRE_WINDOW, landfall)
    d_p, w_p, _, _ = weekly_series(raw.query("place == @name and period == 'prior'"), PRE_WINDOW_PRIOR, landfall - SHIFT)
    w_p["mid"] = w_p["mid"] + SHIFT          # move the reference line onto the typhoon-year dates
    daily[name], weekly[name] = d_e, (w_e, w_p)

    pre_w = w_e.query("week < 0")["pct"].dropna()
    post = w_e.query("week >= 0").set_index("week")["pct"]
    def at(k):
        s = post.reindex([k - 1, k, k + 1]).dropna()
        return round(s.iloc[(s.index.to_series() - k).abs().argmin()]) if len(s) else None
    ok = (post >= RECOVERY_LEVEL)
    streak = ok & ok.shift(-1, fill_value=False)
    low = post.loc[0:4].min()      # lowest visible level in weeks 0-4 (the first weeks are often clouded out)
    if pd.isna(low):
        rec = "no clear data, weeks 0–4"
    elif low >= RECOVERY_LEVEL:
        rec = "no clear outage"
    elif streak.any():
        rec = int(streak.idxmax())
    else:
        rec = f"> {int(post.index.max())}"
    summary.append({"place": name, "baseline days": n_base,
                    "pre-event weekly spread (±%)": round(pre_w.std(), 0),
                    "min, weeks 0–4 (%)": None if pd.isna(low) else round(low),
                    **{f"week {k} (%)": at(k) for k in (2, 4, 8, 12, 16)},
                    f"weeks to ≥{RECOVERY_LEVEL}% (2 wks in a row)": rec})

summary = pd.DataFrame(summary)
summary

In [ ]:
# 9. Small multiples: top = percent of baseline, bottom = clear nights per week
from matplotlib.dates import DateFormatter, MonthLocator

COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
PRIOR = "#a3a29d"

fig, axes = plt.subplots(2, 3, figsize=(15, 6.2), sharex=True, sharey="row",
                         gridspec_kw={"height_ratios": [3, 1]}, constrained_layout=True)
for j, (name, c) in enumerate(zip(PLACES, COLORS)):
    ax, axn = axes[0, j], axes[1, j]
    d, (w, wp) = daily[name], weekly[name]
    ax.axhline(100, color=PRIOR, lw=1, ls="--")
    ax.axhline(RECOVERY_LEVEL, color=PRIOR, lw=1, ls=":")
    ax.axvline(landfall, color=INK2, lw=1)
    ax.scatter(d["date"], d["pct"], s=8, color=c, alpha=0.3, linewidths=0)
    ax.plot(wp["mid"], wp["pct"], color=PRIOR, lw=1.5, label="Same weeks, previous year")
    ax.plot(w["mid"], w["pct"], color=c, lw=2, marker="o", ms=4, label="2013–14 weekly median")
    ax.set_title(name, loc="left", fontsize=11)
    ax.set_ylim(0, 160)
    ax.grid(axis="y", color=GRID, lw=0.8)
    axn.bar(w["mid"], w["days"], width=5, color=c, alpha=0.6)
    axn.axhline(MIN_DAYS_PER_WEEK, color=INK2, lw=0.8, ls=":")
    axn.set_ylim(0, 7)
    axn.xaxis.set_major_locator(MonthLocator(bymonth=[7, 9, 11, 1, 3]))
    axn.xaxis.set_major_formatter(DateFormatter("%b\n%Y"))
    for a in (ax, axn):
        for s in ("top", "right"):
            a.spines[s].set_visible(False)
axes[0, 0].set_ylabel("Percent of pre-event baseline (%)")
axes[1, 0].set_ylabel("Clear days\nper week")
axes[0, 0].annotate("landfall", (landfall, 150), xytext=(4, 0), textcoords="offset points",
                    fontsize=9, color=INK2, va="center")
axes[0, 0].legend(frameon=False, loc="lower left", fontsize=9)
fig.suptitle("Nighttime-light recovery after Typhoon Haiyan: weekly medians of cloud-free observations",
             x=0.01, ha="left", fontsize=12)
fig.text(0.01, -0.03, f"Dots = single clear nights. Dashed = baseline (median {PRE_WINDOW[0]} to {PRE_WINDOW[1]}); "
         f"dotted = {RECOVERY_LEVEL}% recovery threshold. Weeks with < {MIN_DAYS_PER_WEEK} clear days are left blank.",
         fontsize=9, color=INK2)
fig.savefig("figures/fig2_haiyan_recovery_weekly.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 10. Save and download outputs
raw.to_csv("figures/haiyan_daily_raw.csv", index=False)
summary.to_csv("figures/haiyan_recovery_summary.csv", index=False)
from google.colab import files
for f in ["fig1_haiyan_ntl_maps.png", "fig2_haiyan_recovery_weekly.png", "haiyan_recovery_summary.csv"]:
    files.download(f"figures/{f}")

## How to read the results

- **Gray line (same weeks, previous year)**: if it also moves between December and February, that part of the variation is seasonal (clouds, monsoon, holiday lighting), not the typhoon
- **Bars below each panel**: weeks with few clear nights correspond to gaps or less reliable points on the trajectory
- **"Pre-event weekly spread" in the summary table**: this is each city's noise level. A post-event drop has to clearly exceed it to count as a detected outage
- **If Tacloban is still well below 80% by March**, that is likely a real signal: nighttime lights reflect grid repair, household reconnection, rebuilding and people returning combined, not grid energization alone

**What to check**
- How different are the recovery speeds of Tacloban and Ormoc? How do both compare with Cebu City?
- Do the darkest areas on the percent-of-normal map roughly follow Haiyan's track?
- Where are the largest data gaps, and what might cause them (clouds, quality flags)?